In [1]:
import functools
import os
import sys
import traceback
from typing import Dict, Literal, Optional, Tuple
import cellflow
import scanpy as sc
import numpy as np
import functools
from ott.solvers import utils as solver_utils
import optax
from omegaconf import OmegaConf
from typing import NamedTuple, Any
import hydra
import wandb
import anndata as ad
import pandas as pd
import os
from cellflow.training import ComputationCallback
from cellflow.preprocessing import transfer_labels, compute_wknn
from cellflow.training import ComputationCallback
from numpy.typing import ArrayLike
from cellflow.metrics import compute_r_squared, compute_e_distance
from cellflow.metrics import compute_r_squared, compute_e_distance, compute_scalar_mmd, compute_sinkhorn_div
import sys
import pickle
from cellflow.preprocessing import transfer_labels, compute_wknn, centered_pca, project_pca


/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/optuna/study/_optimize.py:29: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from optuna import progress_bar as pbar_module
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWa

In [2]:
def compute_metrics(adata_ref: ad.AnnData, adata_pred: ad.AnnData, donor_deg_dict: dict, adata_ood_true: ad.AnnData, adata_ctrl: ad.AnnData, n_neighbors: int=1, cell_type_col: str = "cell_type_new", min_cells_for_dist_metrics: int = 50) -> dict:
    dict_to_log = {}
    compute_wknn(ref_adata=adata_ref, query_adata=adata_pred, n_neighbors=n_neighbors, ref_rep_key="X_pca", query_rep_key="X_pca_for_ct_transfer")
    transfer_labels(query_adata=adata_pred, ref_adata=adata_ref, label_key=cell_type_col)
    
    e_distance = {}
    r_sq = {}
    mmd = {}
    sdiv_10 = {}
    sdiv_100 = {}
    deg_e_distance = {}
    deg_r_sq = {}
    deg_mmd = {}
    deg_sdiv_10 = {}
    deg_sdiv_100 = {}
    for ct_cyto in donor_deg_dict.keys(): 
        cell_type = ct_cyto.split("_")[1]
        adata_true_ct = adata_ood_true[(adata_ood_true.obs[f"{cell_type_col}"]==cell_type)]
        adata_pred_ct = adata_pred[adata_pred.obs[f"{cell_type_col}_transfer"]==cell_type]
        if adata_pred_ct.n_obs == 0:
            continue
        dist_true_decoded = adata_true_ct.X.toarray()
        dist_pred_decoded = adata_pred_ct.X
        dist_true = adata_true_ct.obsm["X_pca"]
        dist_pred = adata_pred_ct.obsm["X_pca"]
        r_sq[f"decoded_r_squared_{cell_type}"] = compute_r_squared(dist_true_decoded, dist_pred_decoded)
        e_distance[f"e_distance_{cell_type}"] = compute_e_distance(dist_true, dist_pred)
        mmd[f"mmd_{cell_type}"] = compute_scalar_mmd(dist_true, dist_pred)
        sdiv_10[f"div_10_{cell_type}"] = np.nan
        sdiv_100[f"div_100_{cell_type}"] = np.nan

        deg_mask = [True if el in donor_deg_dict[ct_cyto] else False for el in adata_ood_true.var_names]
        deg_true_decoded = adata_true_ct[:,deg_mask].X.toarray()
        deg_pred_decoded = adata_pred_ct[:,deg_mask].X
        deg_r_sq[f"deg_decoded_r_squared_{cell_type}"] = compute_r_squared(deg_true_decoded, deg_pred_decoded)
        deg_e_distance[f"deg_e_distance_{cell_type}"] = compute_e_distance(deg_true_decoded, deg_pred_decoded)
        deg_mmd[f"deg_mmd_{cell_type}"] = compute_scalar_mmd(deg_true_decoded, deg_pred_decoded)
        deg_sdiv_10[f"deg_div_10_{cell_type}"] = np.nan
        deg_sdiv_100[f"deg_div_100_{cell_type}"] = np.nan

    adata_concat = ad.concat([adata_ctrl, adata_pred], join="inner", label="all")
    sc.tl.rank_genes_groups(
            adata_concat,
            groupby="cytokine",
            reference="PBS",
            rankby_abs=True,
            n_genes=50,
            use_raw=False,
            method="wilcoxon",
        )
    predicted_deg_genes = [el[0] for el in list(adata_concat.uns["rank_genes_groups"]["names"])]

    # standard metrics
    decoded_ood_r_squared = compute_r_squared(adata_ood_true.X.toarray(), adata_pred.X)
    ood_e_distance = compute_e_distance(adata_ood_true.obsm["X_pca"], adata_pred.obsm["X_pca"])
    ood_mmd = compute_scalar_mmd(adata_ood_true.obsm["X_pca"], adata_pred.obsm["X_pca"])
    ood_sdiv_10 = np.nan
    ood_sdiv_100 = np.nan
    
    # metrics to return
    dict_to_log["mean_decoded_r_sq_per_cell_type"] = np.mean(list(r_sq.values()))
    dict_to_log["mean_e_distance_per_cell_type"] = np.mean(list(e_distance.values()))
    dict_to_log["mean_mmd_per_cell_type"] = np.mean(list(mmd.values()))
    dict_to_log["mean_sdiv_10_per_cell_type"] = np.mean(list(sdiv_10.values()))
    dict_to_log["mean_sdiv_100_per_cell_type"] = np.mean(list(sdiv_100.values()))
    dict_to_log["mean_deg_r_sq_per_cell_type"] = np.mean(list(deg_r_sq.values()))
    dict_to_log["mean_deg_e_distance_per_cell_type"] = np.mean(list(deg_e_distance.values()))
    dict_to_log["mean_deg_mmd_per_cell_type"] = np.mean(list(deg_mmd.values()))
    dict_to_log["mean_deg_sdiv_10_per_cell_type"] = np.mean(list(deg_sdiv_10.values()))
    dict_to_log["mean_deg_sdiv_100_per_cell_type"] = np.mean(list(deg_sdiv_100.values()))
    
    dict_to_log.update(r_sq)
    dict_to_log.update(e_distance)
    dict_to_log.update(mmd)
    dict_to_log.update(sdiv_10)
    dict_to_log.update(sdiv_100)
    dict_to_log.update(deg_r_sq)
    dict_to_log.update(deg_e_distance)
    dict_to_log.update(deg_mmd)
    dict_to_log.update(deg_sdiv_10)
    dict_to_log.update(deg_sdiv_100)
    dict_to_log["decoded_ood_r_squared"] = decoded_ood_r_squared
    dict_to_log["ood_e_distance"] = ood_e_distance
    dict_to_log["ood_mmd"] = ood_mmd
    dict_to_log["ood_sdiv_10"] = ood_sdiv_10
    dict_to_log["ood_sdiv_100"] = ood_sdiv_100
    dict_to_log["predicted_deg_genes"] = predicted_deg_genes
    return dict_to_log

In [3]:


def get_train_embeddings(adata_same_donor: ad.AnnData, embeddings: dict[str, np.ndarray]) -> dict[str, Any]:
    conds = adata_same_donor.obs.drop_duplicates(subset="condition")
    emb_vectors = {}
    for _,row in conds.iterrows():
        g_1 = row["gene_target_1"]
        g_2  = row["gene_target_2"]
        if g_1=="control" and g_2=="control":
            continue
        elif g_1 != "control" and g_2!= "control":
            emb_vectors[(g_1, g_2)] = (embeddings[g_1] + embeddings[g_2])/2.0
        elif g_1 != "control" and g_2=="control":
            emb_vectors[(g_1, g_2)] = embeddings[g_1]
        elif g_1=="control" and g_2!="control":
            emb_vectors[(g_1, g_2)] = embeddings[g_2]
    return emb_vectors


def get_train_embeddings(adata_train: ad.AnnData, embeddings: dict[str, np.ndarray]) -> dict[str, Any]:
    conds = adata_train.obs.drop_duplicates(subset="cytokine")
    emb_vectors = {}
    for _,row in conds.iterrows():
        cyto = row["cytokine"]
        if cyto=="PBS":
            continue
        emb_vectors[cyto] = embeddings[cyto]
    return emb_vectors

def find_closest_embedding(emb_0: np.ndarray, reference_embeddings: dict[str, np.ndarray]) -> Tuple[str, ...]:
    closest_emb = None
    closest_dist = np.inf
    for ref, ref_emb in reference_embeddings.items():
        dist = np.sum((emb_0-ref_emb)**2)
        if dist < closest_dist:
            closest_dist = dist
            closest_emb = ref
    return closest_emb



In [4]:
donor = "Donor1"
cytokine_held_out = "IL-32"
condition = f"{donor}_{cytokine_held_out}"

In [5]:
#with open("/lustre/groups/ml01/workspace/ot_perturbation/data/embeddings/pbmc_cytokine_mashup.pkl", "rb") as pickle_file:
#        mashup_embeddings = pickle.load(pickle_file)

In [6]:

out_dir_closest_embedding = "/lustre/groups/ml01/workspace/ot_perturbation/models/additive_model/pbmc_new_cytokine/closest_embedding"
cytokine_held_out = "TWEAK"
idx_given_cytokine = "0"

control_key = "is_control"

adata_base = sc.read_h5ad(f"/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/new_cytokine/adata_base_{cytokine_held_out}.h5ad")
adata_rest = sc.read_h5ad(f"/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/new_cytokine/adata_rest_{cytokine_held_out}.h5ad")
donors_to_impute = adata_rest.uns["split_info"][idx_given_cytokine]["donors_to_impute"]
donors_to_train_data = adata_rest.uns["split_info"][idx_given_cytokine]["donors_to_train_data"]
adata_to_append = adata_rest[adata_rest.obs["donor"].isin(donors_to_train_data)]
adata_train = ad.concat((adata_base, adata_to_append))
adata_ctrl = adata_train[adata_train.obs[control_key].to_numpy()]

In [9]:
adata_base.uns.keys()

dict_keys(['donor_one_hot', 'esm2_embeddings', 'hvg', 'log1p'])

In [10]:
mashup_embeddings = adata_base.uns["esm2_embeddings"]

In [11]:



with open("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/idcs_to_keep.pkl", "rb") as pickle_file:
    idcs_to_keep = pickle.load(pickle_file)
adata_full = sc.read_h5ad("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/pbmc_with_pca.h5ad")
adata_ref = adata_full[adata_full.obs_names.isin(idcs_to_keep)]
with open("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/degs.pkl", "rb") as pickle_file:
    deg_genes = pickle.load(pickle_file)




for donor in donors_to_impute:
    adata_ctrl_current_donor = adata_ctrl[adata_ctrl.obs["donor"]==donor]
    if adata_ctrl_current_donor.n_obs > 10000:
        sc.pp.subsample(adata_ctrl_current_donor, n_obs=10000)
        
    train_embeddings = get_train_embeddings(adata_train[adata_train.obs["donor"]==donor], mashup_embeddings)
    closest_embedding = find_closest_embedding(mashup_embeddings[cytokine_held_out], train_embeddings)

    adata_pred = adata_train[(adata_train.obs["donor"]==donor) & (adata_train.obs["cytokine"]==closest_embedding)]
    adata_pred.uns["donors_in_train"] = list(adata_to_append.obs["donor"].unique())    
    adata_pred.obs["cytokine"] = cytokine_held_out
    adata_pred.obs["donor"] = donor
    adata_pred.X = adata_pred.X.toarray()

    
    if adata_pred.n_obs > 10000:
        sc.pp.subsample(adata_pred, n_obs=10000)
    
    project_pca(query_adata=adata_pred, ref_adata=adata_ref, obsm_key_added="X_pca_for_ct_transfer")
    project_pca(query_adata=adata_pred, ref_adata=adata_full, obsm_key_added="X_pca")
    cond_orig = condition
    condition = condition + "_" + str(len(adata_pred.uns["donors_in_train"]))
    donor_deg_dict = {k: v for k, v in deg_genes.items() if (k.startswith(donor) and k.endswith(f"_{cytokine_held_out}"))}
    adata_ood_true = adata_full[(adata_full.obs["donor"] == donor) & (adata_full.obs["cytokine"]==cytokine_held_out)]
    
    out = compute_metrics(adata_ref=adata_ref, adata_pred=adata_pred, donor_deg_dict=donor_deg_dict, adata_ood_true=adata_ood_true, adata_ctrl=adata_ctrl_current_donor)
    pd.DataFrame.from_dict(out, columns=[condition], orient="index").to_csv(os.path.join(out_dir_closest_embedding, f"{idx_given_cytokine}_{condition}.csv"))
    

    break

/tmp/ipykernel_1623447/614052076.py:20: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata_pred.uns["donors_in_train"] = list(adata_to_append.obs["donor"].unique())
/ictstr01/home/icb/dominik.klein/git_repos/cell_flow_perturbation/src/cellflow/preprocessing/_wknn.py:88: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  ref_adata.uns[uns_key_added] = wknn


OSError: Cannot save file into a non-existent directory: '/lustre/groups/ml01/workspace/ot_perturbation/models/additive_model/pbmc_new_cytokine/closest_embedding'

In [39]:
pd.DataFrame.from_dict(out, columns=[condition], orient="index")

,Donor6_TWEAK_0_0_0
mean_decoded_r_sq_per_cell_type,0.966032
mean_e_distance_per_cell_type,16.37118
mean_mmd_per_cell_type,0.011567
mean_sdiv_10_per_cell_type,NaN
mean_sdiv_100_per_cell_type,NaN
...,...
ood_e_distance,3.73728
ood_mmd,0.000706
ood_sdiv_10,NaN
ood_sdiv_100,NaN


In [43]:
adata_train[adata_train.obs["is_control"]].obs["cytokine"]

91_103_005__s1      PBS
91_103_059__s1      PBS
91_115_049__s1      PBS
91_115_074__s1      PBS
91_115_107__s1      PBS
                   ... 
96_185_071__s144    PBS
96_186_014__s144    PBS
96_186_064__s144    PBS
96_186_097__s144    PBS
96_186_150__s144    PBS
Name: cytokine, Length: 629701, dtype: category
Categories (1, object): ['PBS']

In [35]:
adata_ctrl = adata_ctrl_current_donor
adata_pred.X = adata_pred.X.toarray()
dict_to_log = {}
n_neighbors=1
cell_type_col= "cell_type_new"
min_cells_for_dist_metrics: int = 50

compute_wknn(ref_adata=adata_ref, query_adata=adata_pred, n_neighbors=n_neighbors, ref_rep_key="X_pca", query_rep_key="X_pca_for_ct_transfer")
transfer_labels(query_adata=adata_pred, ref_adata=adata_ref, label_key=cell_type_col)

e_distance = {}
r_sq = {}
mmd = {}
sdiv_10 = {}
sdiv_100 = {}
deg_e_distance = {}
deg_r_sq = {}
deg_mmd = {}
deg_sdiv_10 = {}
deg_sdiv_100 = {}
for ct_cyto in donor_deg_dict.keys(): 
    cell_type = ct_cyto.split("_")[1]
    adata_true_ct = adata_ood_true[(adata_ood_true.obs[f"{cell_type_col}"]==cell_type)]
    adata_pred_ct = adata_pred[adata_pred.obs[f"{cell_type_col}_transfer"]==cell_type]
    if adata_pred_ct.n_obs == 0:
        continue
    dist_true_decoded = adata_true_ct.X.toarray()
    dist_pred_decoded = adata_pred_ct.X
    dist_true = adata_true_ct.obsm["X_pca"]
    dist_pred = adata_pred_ct.obsm["X_pca"]

In [33]:
dist_true_decoded.shape

(283, 2000)

In [34]:
dist_pred_decoded

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 30416 stored elements and shape (384, 2000)>

In [36]:
r_sq[f"decoded_r_squared_{cell_type}"] = compute_r_squared(dist_true_decoded, dist_pred_decoded)
e_distance[f"e_distance_{cell_type}"] = compute_e_distance(dist_true, dist_pred)

In [18]:
donors_to_train_data

array([], dtype=float64)

In [19]:
train_embeddings = get_train_embeddings(adata_train[adata_train.obs["donor"].isin(donors_to_train_data)], mashup_embeddings)


In [24]:
adata_train.obs["cytokine"].value_counts()["TWEAK"]

KeyError: 'TWEAK'

In [16]:
dists = {}
emb_0 = mashup_embeddings["TWEAK"]
for cyto, emb in train_embeddings.items():
    dists[cyto] = np.sum((emb_0-emb)**2)

In [17]:
dists

{'4-1BBL': 6.323430618861442,
 'ADSF': 17.06759153313781,
 'APRIL': 14.936906486878579,
 'BAFF': 11.513200375015046,
 'C3a': 23.16079295282893,
 'C5a': 18.074608430045767,
 'CD27L': 5.495880229024266,
 'CD30L': 9.395657124947892,
 'CD40L': 20.269207084402478,
 'CT-1': 16.6012699928647,
 'Decorin': 27.451054094179316,
 'EGF': 24.75004862439573,
 'EPO': 20.234594734149283,
 'FGF-beta': 22.436448969188376,
 'FLT3L': 18.757741914242096,
 'FasL': 22.15804755536773,
 'G-CSF': 16.176751262009265,
 'GDNF': 19.80370566603663,
 'GITRL': 8.683724526754364,
 'GM-CSF': 17.34400293839041,
 'HGF': 21.432901034654584,
 'IFN-alpha1': 12.849684134803343,
 'IFN-beta': 20.34037485083698,
 'IFN-epsilon': 17.078502131193233,
 'IFN-gamma': 20.750390320073723,
 'IFN-lambda1': 15.59363767043136,
 'IFN-lambda2': 16.216449551135767,
 'IFN-lambda3': 17.097294616466783,
 'IFN-omega': 21.896169036926512,
 'IGF-1': 24.78325216369006,
 'IL-1-alpha': 20.286465009011835,
 'IL-1-beta': 20.268223978571598,
 'IL-10': 19.5

In [46]:
donor_held_out = "Donor1"
idx_given_donor = "7"
adata_train = sc.read_h5ad(f"/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/new_donor/{donor_held_out}/{str(idx_given_donor)}/adata_train_{donor_held_out}.h5ad")
adata_ood_perturbed  = sc.read_h5ad(f"/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/new_donor/{donor_held_out}/{str(idx_given_donor)}/adata_ood_{donor_held_out}.h5ad")
cytokines_to_impute = adata_train.uns["split_info"][idx_given_donor]["cytokines_to_impute"]
cytokines_to_train_data = adata_train.uns["split_info"][idx_given_donor]["cytokines_to_train_data"]


In [47]:
cytokines_to_train_data

array(['IL-3', 'C5a', 'PBS'], dtype=object)

In [45]:
adata_train.uns["split_info"]

{'0': {'cytokines_to_impute': array(['OX40L', 'IL-32-beta', 'IL-1Ra', 'IFN-gamma', 'IFN-omega', 'BAFF',
         'CD27L', 'ADSF', 'FasL', 'M-CSF'], dtype=object),
  'cytokines_to_train_data': array(['IL-8', 'PBS'], dtype=object)},
 '1': {'cytokines_to_impute': array(['OX40L', 'IL-32-beta', 'IL-1Ra', 'IFN-gamma', 'IFN-omega', 'BAFF',
         'CD27L', 'ADSF', 'FasL', 'M-CSF'], dtype=object),
  'cytokines_to_train_data': array(['IFN-alpha1', 'PBS'], dtype=object)},
 '2': {'cytokines_to_impute': array(['OX40L', 'IL-32-beta', 'IL-1Ra', 'IFN-gamma', 'IFN-omega', 'BAFF',
         'CD27L', 'ADSF', 'FasL', 'M-CSF'], dtype=object),
  'cytokines_to_train_data': array(['IFN-lambda3', 'PBS'], dtype=object)},
 '3': {'cytokines_to_impute': array(['OX40L', 'IL-32-beta', 'IL-1Ra', 'IFN-gamma', 'IFN-omega', 'BAFF',
         'CD27L', 'ADSF', 'FasL', 'M-CSF'], dtype=object),
  'cytokines_to_train_data': array(['IL-17C', 'IFN-beta', 'TSLP', 'GITRL', 'LIGHT', 'IL-26', 'IL-17A',
         'TNF-alpha', 'IL-1-

In [49]:
adata_train[adata_train.obs["donor"]==donor_held_out].obs["cytokine"].nunique()

3


# New donor

In [1]:
import functools
import os
import sys
import traceback
from typing import Dict, Literal, Optional, Tuple
import cellflow
import scanpy as sc
import numpy as np
import functools
from ott.solvers import utils as solver_utils
import optax
from omegaconf import OmegaConf
from typing import NamedTuple, Any
import hydra
import wandb
import anndata as ad
import pandas as pd
import os
from cellflow.training import ComputationCallback
from cellflow.preprocessing import transfer_labels, compute_wknn
from cellflow.training import ComputationCallback
from numpy.typing import ArrayLike
from cellflow.metrics import compute_r_squared, compute_e_distance
from cellflow.metrics import compute_r_squared, compute_e_distance, compute_scalar_mmd, compute_sinkhorn_div
import sys
import pickle
from cellflow.preprocessing import transfer_labels, compute_wknn, centered_pca, project_pca



/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/optuna/study/_optimize.py:29: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from optuna import progress_bar as pbar_module
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWa

In [2]:


def compute_metrics(adata_ref: ad.AnnData, adata_pred: ad.AnnData, donor_deg_dict: dict, adata_ood_true: ad.AnnData, adata_ctrl: ad.AnnData, n_neighbors: int=1, cell_type_col: str = "cell_type_new", min_cells_for_dist_metrics: int = 50) -> dict:
    dict_to_log = {}
    compute_wknn(ref_adata=adata_ref, query_adata=adata_pred, n_neighbors=n_neighbors, ref_rep_key="X_pca", query_rep_key="X_pca_for_ct_transfer")
    transfer_labels(query_adata=adata_pred, ref_adata=adata_ref, label_key=cell_type_col)
    
    e_distance = {}
    r_sq = {}
    mmd = {}
    sdiv_10 = {}
    sdiv_100 = {}
    deg_e_distance = {}
    deg_r_sq = {}
    deg_mmd = {}
    deg_sdiv_10 = {}
    deg_sdiv_100 = {}
    for ct_cyto in donor_deg_dict.keys(): 
        cell_type = ct_cyto.split("_")[1]
        adata_true_ct = adata_ood_true[(adata_ood_true.obs[f"{cell_type_col}"]==cell_type)]
        adata_pred_ct = adata_pred[adata_pred.obs[f"{cell_type_col}_transfer"]==cell_type]
        if adata_pred_ct.n_obs == 0:
            continue
        dist_true_decoded = adata_true_ct.X.toarray()
        dist_pred_decoded = adata_pred_ct.X
        dist_true = adata_true_ct.obsm["X_pca"]
        dist_pred = adata_pred_ct.obsm["X_pca"]
        r_sq[f"decoded_r_squared_{cell_type}"] = compute_r_squared(dist_true_decoded, dist_pred_decoded)
        e_distance[f"e_distance_{cell_type}"] = compute_e_distance(dist_true, dist_pred)
        mmd[f"mmd_{cell_type}"] = compute_scalar_mmd(dist_true, dist_pred)
        sdiv_10[f"div_10_{cell_type}"] = np.nan
        sdiv_100[f"div_100_{cell_type}"] = np.nan

        deg_mask = [True if el in donor_deg_dict[ct_cyto] else False for el in adata_ood_true.var_names]
        deg_true_decoded = adata_true_ct[:,deg_mask].X.toarray()
        deg_pred_decoded = adata_pred_ct[:,deg_mask].X
        deg_r_sq[f"deg_decoded_r_squared_{cell_type}"] = compute_r_squared(deg_true_decoded, deg_pred_decoded)
        deg_e_distance[f"deg_e_distance_{cell_type}"] = compute_e_distance(deg_true_decoded, deg_pred_decoded)
        deg_mmd[f"deg_mmd_{cell_type}"] = compute_scalar_mmd(deg_true_decoded, deg_pred_decoded)
        deg_sdiv_10[f"deg_div_10_{cell_type}"] = np.nan
        deg_sdiv_100[f"deg_div_100_{cell_type}"] = np.nan

    adata_concat = ad.concat([adata_ctrl, adata_pred], join="inner", label="all")
    sc.tl.rank_genes_groups(
            adata_concat,
            groupby="cytokine",
            reference="PBS",
            rankby_abs=True,
            n_genes=50,
            use_raw=False,
            method="wilcoxon",
        )
    predicted_deg_genes = [el[0] for el in list(adata_concat.uns["rank_genes_groups"]["names"])]

    # standard metrics
    decoded_ood_r_squared = compute_r_squared(adata_ood_true.X.toarray(), adata_pred.X)
    ood_e_distance = compute_e_distance(adata_ood_true.obsm["X_pca"], adata_pred.obsm["X_pca"])
    ood_mmd = compute_scalar_mmd(adata_ood_true.obsm["X_pca"], adata_pred.obsm["X_pca"])
    ood_sdiv_10 = np.nan
    ood_sdiv_100 = np.nan

    # metrics to return
    dict_to_log["mean_decoded_r_sq_per_cell_type"] = np.mean(list(r_sq.values()))
    dict_to_log["mean_e_distance_per_cell_type"] = np.mean(list(e_distance.values()))
    dict_to_log["mean_mmd_per_cell_type"] = np.mean(list(mmd.values()))
    dict_to_log["mean_sdiv_10_per_cell_type"] = np.mean(list(sdiv_10.values()))
    dict_to_log["mean_sdiv_100_per_cell_type"] = np.mean(list(sdiv_100.values()))
    dict_to_log["mean_deg_r_sq_per_cell_type"] = np.mean(list(deg_r_sq.values()))
    dict_to_log["mean_deg_e_distance_per_cell_type"] = np.mean(list(deg_e_distance.values()))
    dict_to_log["mean_deg_mmd_per_cell_type"] = np.mean(list(deg_mmd.values()))
    dict_to_log["mean_deg_sdiv_10_per_cell_type"] = np.mean(list(deg_sdiv_10.values()))
    dict_to_log["mean_deg_sdiv_100_per_cell_type"] = np.mean(list(deg_sdiv_100.values()))
    
    dict_to_log.update(r_sq)
    dict_to_log.update(e_distance)
    dict_to_log.update(mmd)
    dict_to_log.update(sdiv_10)
    dict_to_log.update(sdiv_100)
    dict_to_log.update(deg_r_sq)
    dict_to_log.update(deg_e_distance)
    dict_to_log.update(deg_mmd)
    dict_to_log.update(deg_sdiv_10)
    dict_to_log.update(deg_sdiv_100)
    dict_to_log["decoded_ood_r_squared"] = decoded_ood_r_squared
    dict_to_log["ood_e_distance"] = ood_e_distance
    dict_to_log["ood_mmd"] = ood_mmd
    dict_to_log["ood_sdiv_10"] = ood_sdiv_10
    dict_to_log["ood_sdiv_100"] = ood_sdiv_100
    dict_to_log["predicted_deg_genes"] = predicted_deg_genes
    return dict_to_log


def get_train_embeddings(adata_train: ad.AnnData, embeddings: dict[str, np.ndarray]) -> dict[str, Any]:
    conds = adata_train.obs.drop_duplicates(subset="cytokine")
    emb_vectors = {}
    for _,row in conds.iterrows():
        cyto = row["cytokine"]
        if cyto=="PBS":
            continue
        emb_vectors[cyto] = embeddings[cyto]
    return emb_vectors

def find_closest_embedding(emb_0: np.ndarray, reference_embeddings: dict[str, np.ndarray]) -> Tuple[str, ...]:
    closest_emb = None
    closest_dist = np.inf
    for ref, ref_emb in reference_embeddings.items():
        dist = np.sum((emb_0-ref_emb)**2)
        if dist < closest_dist:
            closest_dist = dist
            closest_emb = ref
    return closest_emb
    

        



In [3]:

out_dir = "/lustre/groups/ml01/workspace/ot_perturbation/models/additive_model/pbmc_new_donor/closest_embedding"
donor_held_out = "Donor1"
idx_given_donor = "3"

control_key = "is_control"

adata_train = sc.read_h5ad(f"/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/new_donor/{donor_held_out}/{str(idx_given_donor)}/adata_train_{donor_held_out}.h5ad")
adata_ood_perturbed  = sc.read_h5ad(f"/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/new_donor/{donor_held_out}/{str(idx_given_donor)}/adata_ood_{donor_held_out}.h5ad")
cytokines_to_impute = adata_train.uns["split_info"][idx_given_donor]["cytokines_to_impute"]
cytokines_to_train_data = adata_train.uns["split_info"][idx_given_donor]["cytokines_to_train_data"]

if len(cytokines_to_train_data) == 0:
    sys.exit(0)

adata_ctrl = adata_train[adata_train.obs[control_key].to_numpy()]


with open("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/idcs_to_keep.pkl", "rb") as pickle_file:
    idcs_to_keep = pickle.load(pickle_file)
adata_full = sc.read_h5ad("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/pbmc_with_pca.h5ad")
adata_ref = adata_full[adata_full.obs_names.isin(idcs_to_keep)]
with open("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/degs.pkl", "rb") as pickle_file:
    deg_genes = pickle.load(pickle_file)

with open("/lustre/groups/ml01/workspace/ot_perturbation/data/embeddings/pbmc_cytokine_mashup.pkl", "rb") as pickle_file:
    mashup_embeddings = pickle.load(pickle_file)

adata_ctrl_current_donor = adata_ctrl[adata_ctrl.obs["donor"]==donor_held_out]
if adata_ctrl_current_donor.n_obs > 10000:
        sc.pp.subsample(adata_ctrl_current_donor, n_obs=10000)
    
for cytokine in cytokines_to_impute:
    train_cytokines_same_donor = {k:v for k,v in mashup_embeddings.items() if k in cytokines_to_train_data and k != "PBS"}
    closest_cyto = find_closest_embedding(mashup_embeddings[cytokine], train_cytokines_same_donor)
    adata_pred = adata_train[(adata_train.obs["donor"]==donor_held_out)&(adata_train.obs["cytokine"]==closest_cyto)].copy()
    adata_pred.X = adata_pred.X.toarray()
    
    condition = f"{donor_held_out}_{cytokine}"
    
    project_pca(query_adata=adata_pred, ref_adata=adata_ref, obsm_key_added="X_pca_for_ct_transfer")
    project_pca(query_adata=adata_pred, ref_adata=adata_full, obsm_key_added="X_pca")
    cond_orig = condition
    condition = condition + "_" + str(len(cytokines_to_train_data))
    donor_deg_dict = {k: v for k, v in deg_genes.items() if (k.startswith(donor_held_out) and k.endswith(f"_{cytokine}"))}
    adata_ood_true = adata_full[(adata_full.obs["donor"] == donor_held_out) & (adata_full.obs["cytokine"]==cytokine)]
    
    out = compute_metrics(adata_ref=adata_ref, adata_pred=adata_pred, donor_deg_dict=donor_deg_dict, adata_ood_true=adata_ood_true, adata_ctrl=adata_ctrl_current_donor)
    out["num_cytokines_in_train"] = len(cytokines_to_train_data)
    pd.DataFrame.from_dict(out, columns=[condition], orient="index").to_csv(os.path.join(out_dir, f"{idx_given_donor}_{condition}.csv"))


AttributeError: 'str' object has no attribute 'shape'

In [6]:
closest_cyto = find_closest_embedding(mashup_embeddings[cytokine], train_cytokines_same_donor)
adata_pred = adata_train[(adata_train.obs["donor"]==donor_held_out)&(adata_train.obs["cytokine"]==closest_cyto)]
    

In [ ]:
    
for cytokine in cytokines_to_impute:
    print(cytokine)
    train_cytokines_same_donor = {k:v for k,v in mashup_embeddings.items() if k in cytokines_to_train_data and k != "PBS"}
    closest_cyto = find_closest_embedding(mashup_embeddings[cytokine], train_cytokines_same_donor)
    adata_pred = adata_train[(adata_train.obs["donor"]==donor_held_out)&(adata_train.obs["cytokine"]==closest_cyto)].copy()
    adata_pred.X = adata_pred.X.toarray()
    
    condition = f"{donor_held_out}_{cytokine}"
    
    project_pca(query_adata=adata_pred, ref_adata=adata_ref, obsm_key_added="X_pca_for_ct_transfer")
    project_pca(query_adata=adata_pred, ref_adata=adata_full, obsm_key_added="X_pca")
    cond_orig = condition
    condition = condition + "_" + str(len(cytokines_to_train_data))
    donor_deg_dict = {k: v for k, v in deg_genes.items() if (k.startswith(donor_held_out) and k.endswith(f"_{cytokine}"))}
    adata_ood_true = adata_full[(adata_full.obs["donor"] == donor_held_out) & (adata_full.obs["cytokine"]==cytokine)]
    
    out = compute_metrics(adata_ref=adata_ref, adata_pred=adata_pred, donor_deg_dict=donor_deg_dict, adata_ood_true=adata_ood_true, adata_ctrl=adata_ctrl_current_donor)
    out["num_cytokines_in_train"] = len(cytokines_to_train_data)
    pd.DataFrame.from_dict(out, columns=[condition], orient="index").to_csv(os.path.join(out_dir, f"{idx_given_donor}_{condition}.csv"))


OX40L


In [18]:
adata_pred.X = adata_pred.X.toarray()

In [19]:
adata_pred.X

array([[0.       , 0.       , 0.       , ..., 0.       , 3.3515942,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 2.9873166,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       ...,
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        2.1065965],
       [0.       , 0.       , 0.       , ..., 0.       , 3.680338 ,
        2.6309242]], dtype=float32)

In [1]:
import functools
import os
import sys
import traceback
from typing import Dict, Literal, Optional, Tuple
import cellflow
import scanpy as sc
import numpy as np
import functools
from ott.solvers import utils as solver_utils
import optax
from omegaconf import OmegaConf
from typing import NamedTuple, Any
import hydra
import wandb
import anndata as ad
import pandas as pd
import os
from cellflow.training import ComputationCallback
from cellflow.preprocessing import transfer_labels, compute_wknn
from cellflow.training import ComputationCallback
from numpy.typing import ArrayLike
from cellflow.metrics import compute_r_squared, compute_e_distance
from cellflow.metrics import compute_r_squared, compute_e_distance, compute_scalar_mmd, compute_sinkhorn_div
import sys
import pickle
from cellflow.preprocessing import transfer_labels, compute_wknn, centered_pca, project_pca

/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/optuna/study/_optimize.py:29: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from optuna import progress_bar as pbar_module
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWa

In [2]:

def compute_metrics(adata_ref: ad.AnnData, adata_pred: ad.AnnData, donor_deg_dict: dict, adata_ood_true: ad.AnnData, adata_ctrl: ad.AnnData, n_neighbors: int=1, cell_type_col: str = "cell_type_new", min_cells_for_dist_metrics: int = 50) -> dict:
    dict_to_log = {}
    compute_wknn(ref_adata=adata_ref, query_adata=adata_pred, n_neighbors=n_neighbors, ref_rep_key="X_pca", query_rep_key="X_pca_for_ct_transfer")
    transfer_labels(query_adata=adata_pred, ref_adata=adata_ref, label_key=cell_type_col)
    
    e_distance = {}
    r_sq = {}
    mmd = {}
    sdiv_10 = {}
    sdiv_100 = {}
    deg_e_distance = {}
    deg_r_sq = {}
    deg_mmd = {}
    deg_sdiv_10 = {}
    deg_sdiv_100 = {}
    for ct_cyto in donor_deg_dict.keys(): 
        cell_type = ct_cyto.split("_")[1]
        adata_true_ct = adata_ood_true[(adata_ood_true.obs[f"{cell_type_col}"]==cell_type)]
        adata_pred_ct = adata_pred[adata_pred.obs[f"{cell_type_col}_transfer"]==cell_type]
        if adata_pred_ct.n_obs == 0:
            continue
        dist_true_decoded = adata_true_ct.X.toarray()
        dist_pred_decoded = adata_pred_ct.X
        dist_true = adata_true_ct.obsm["X_pca"]
        dist_pred = adata_pred_ct.obsm["X_pca"]
        r_sq[f"decoded_r_squared_{cell_type}"] = compute_r_squared(dist_true_decoded, dist_pred_decoded)
        e_distance[f"e_distance_{cell_type}"] = compute_e_distance(dist_true, dist_pred)
        mmd[f"mmd_{cell_type}"] = compute_scalar_mmd(dist_true, dist_pred)
        sdiv_10[f"div_10_{cell_type}"] = np.nan
        sdiv_100[f"div_100_{cell_type}"] = np.nan

        deg_mask = [True if el in donor_deg_dict[ct_cyto] else False for el in adata_ood_true.var_names]
        deg_true_decoded = adata_true_ct[:,deg_mask].X.toarray()
        deg_pred_decoded = adata_pred_ct[:,deg_mask].X
        deg_r_sq[f"deg_decoded_r_squared_{cell_type}"] = compute_r_squared(deg_true_decoded, deg_pred_decoded)
        deg_e_distance[f"deg_e_distance_{cell_type}"] = compute_e_distance(deg_true_decoded, deg_pred_decoded)
        deg_mmd[f"deg_mmd_{cell_type}"] = compute_scalar_mmd(deg_true_decoded, deg_pred_decoded)
        deg_sdiv_10[f"deg_div_10_{cell_type}"] = np.nan
        deg_sdiv_100[f"deg_div_100_{cell_type}"] = np.nan

    adata_concat = ad.concat([adata_ctrl, adata_pred], join="inner", label="all")
    sc.tl.rank_genes_groups(
            adata_concat,
            groupby="cytokine",
            reference="PBS",
            rankby_abs=True,
            n_genes=50,
            use_raw=False,
            method="wilcoxon",
        )
    predicted_deg_genes = [el[0] for el in list(adata_concat.uns["rank_genes_groups"]["names"])]

    # standard metrics
    decoded_ood_r_squared = compute_r_squared(adata_ood_true.X.toarray(), adata_pred.X)
    ood_e_distance = compute_e_distance(adata_ood_true.obsm["X_pca"], adata_pred.obsm["X_pca"])
    ood_mmd = compute_scalar_mmd(adata_ood_true.obsm["X_pca"], adata_pred.obsm["X_pca"])
    ood_sdiv_10 = np.nan
    ood_sdiv_100 = np.nan

    # metrics to return
    dict_to_log["mean_decoded_r_sq_per_cell_type"] = np.mean(list(r_sq.values()))
    dict_to_log["mean_e_distance_per_cell_type"] = np.mean(list(e_distance.values()))
    dict_to_log["mean_mmd_per_cell_type"] = np.mean(list(mmd.values()))
    dict_to_log["mean_sdiv_10_per_cell_type"] = np.mean(list(sdiv_10.values()))
    dict_to_log["mean_sdiv_100_per_cell_type"] = np.mean(list(sdiv_100.values()))
    dict_to_log["mean_deg_r_sq_per_cell_type"] = np.mean(list(deg_r_sq.values()))
    dict_to_log["mean_deg_e_distance_per_cell_type"] = np.mean(list(deg_e_distance.values()))
    dict_to_log["mean_deg_mmd_per_cell_type"] = np.mean(list(deg_mmd.values()))
    dict_to_log["mean_deg_sdiv_10_per_cell_type"] = np.mean(list(deg_sdiv_10.values()))
    dict_to_log["mean_deg_sdiv_100_per_cell_type"] = np.mean(list(deg_sdiv_100.values()))
    
    dict_to_log.update(r_sq)
    dict_to_log.update(e_distance)
    dict_to_log.update(mmd)
    dict_to_log.update(sdiv_10)
    dict_to_log.update(sdiv_100)
    dict_to_log.update(deg_r_sq)
    dict_to_log.update(deg_e_distance)
    dict_to_log.update(deg_mmd)
    dict_to_log.update(deg_sdiv_10)
    dict_to_log.update(deg_sdiv_100)
    dict_to_log["decoded_ood_r_squared"] = decoded_ood_r_squared
    dict_to_log["ood_e_distance"] = ood_e_distance
    dict_to_log["ood_mmd"] = ood_mmd
    dict_to_log["ood_sdiv_10"] = ood_sdiv_10
    dict_to_log["ood_sdiv_100"] = ood_sdiv_100
    dict_to_log["predicted_deg_genes"] = predicted_deg_genes
    return dict_to_log


def get_train_embeddings(adata_train: ad.AnnData, embeddings: dict[str, np.ndarray]) -> dict[str, Any]:
    conds = adata_train.obs.drop_duplicates(subset="cytokine")
    emb_vectors = {}
    for _,row in conds.iterrows():
        cyto = row["cytokine"]
        if cyto=="PBS":
            continue
        emb_vectors[cyto] = embeddings[cyto]
    return emb_vectors

def find_closest_embedding(emb_0: np.ndarray, reference_embeddings: dict[str, np.ndarray]) -> Tuple[str, ...]:
    closest_emb = None
    closest_dist = np.inf
    for ref, ref_emb in reference_embeddings.items():
        dist = np.sum((emb_0-ref_emb)**2)
        if dist < closest_dist:
            closest_dist = dist
            closest_emb = ref
    return closest_emb
    


In [ ]:
out_dir = "/lustre/groups/ml01/workspace/ot_perturbation/models/additive_model/pbmc_new_donor/closest_embedding"
donor_held_out = "Donor2"
idx_given_donor = "5"

control_key = "is_control"
    
adata_train = sc.read_h5ad(f"/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/new_donor/{donor_held_out}/{str(idx_given_donor)}/adata_train_{donor_held_out}.h5ad")
adata_ood_perturbed  = sc.read_h5ad(f"/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/new_donor/{donor_held_out}/{str(idx_given_donor)}/adata_ood_{donor_held_out}.h5ad")
cytokines_to_impute = adata_train.uns["split_info"][idx_given_donor]["cytokines_to_impute"]
cytokines_to_train_data = adata_train.uns["split_info"][idx_given_donor]["cytokines_to_train_data"]

if len(cytokines_to_train_data) == 0:
    sys.exit(0)

adata_ctrl = adata_train[adata_train.obs[control_key].to_numpy()]


with open("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/idcs_to_keep.pkl", "rb") as pickle_file:
    idcs_to_keep = pickle.load(pickle_file)
adata_full = sc.read_h5ad("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/pbmc_with_pca.h5ad")
adata_ref = adata_full[adata_full.obs_names.isin(idcs_to_keep)]
with open("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/degs.pkl", "rb") as pickle_file:
    deg_genes = pickle.load(pickle_file)

with open("/lustre/groups/ml01/workspace/ot_perturbation/data/embeddings/pbmc_cytokine_mashup.pkl", "rb") as pickle_file:
    mashup_embeddings = pickle.load(pickle_file)

adata_ctrl_current_donor = adata_ctrl[adata_ctrl.obs["donor"]==donor_held_out]
if adata_ctrl_current_donor.n_obs > 10000:
        sc.pp.subsample(adata_ctrl_current_donor, n_obs=10000)
    
for cytokine in cytokines_to_impute:
    print(cytokine)
    train_cytokines_same_donor = {k:v for k,v in mashup_embeddings.items() if k in cytokines_to_train_data and k != "PBS"}
    closest_cyto = find_closest_embedding(mashup_embeddings[cytokine], train_cytokines_same_donor)
    adata_pred = adata_train[(adata_train.obs["donor"]==donor_held_out)&(adata_train.obs["cytokine"]==closest_cyto)].copy()
    adata_pred.X = adata_pred.X.toarray()
    
    condition = f"{donor_held_out}_{cytokine}"
    
    project_pca(query_adata=adata_pred, ref_adata=adata_ref, obsm_key_added="X_pca_for_ct_transfer")
    project_pca(query_adata=adata_pred, ref_adata=adata_full, obsm_key_added="X_pca")
    cond_orig = condition
    condition = condition + "_" + str(len(cytokines_to_train_data))
    donor_deg_dict = {k: v for k, v in deg_genes.items() if (k.startswith(donor_held_out) and k.endswith(f"_{cytokine}"))}
    adata_ood_true = adata_full[(adata_full.obs["donor"] == donor_held_out) & (adata_full.obs["cytokine"]==cytokine)]
    
    out = compute_metrics(adata_ref=adata_ref, adata_pred=adata_pred, donor_deg_dict=donor_deg_dict, adata_ood_true=adata_ood_true, adata_ctrl=adata_ctrl_current_donor)
    out["num_cytokines_in_train"] = len(cytokines_to_train_data)
    pd.DataFrame.from_dict(out, columns=[condition], orient="index").to_csv(os.path.join(out_dir, f"{idx_given_donor}_{condition}.csv"))


OX40L


/ictstr01/home/icb/dominik.klein/git_repos/cell_flow_perturbation/src/cellflow/preprocessing/_wknn.py:88: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  ref_adata.uns[uns_key_added] = wknn
